In [0]:
from pyspark.sql.types import  StructType, StructField, IntegerType, StringType,DateType


In [0]:
customer_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("telephone", StringType(), True),
    StructField("email", StringType(), True)
     ])
    

In [0]:
customer_df = (spark.readStream
               .format("csv")
               .schema(customer_schema)
               .load("/Volumes/udemydatabricks/dataengg/streaming_volume/customer_stream/"))


In [0]:
from pyspark.sql.functions import col,current_timestamp
customer_transformed_df = (
                            customer_df.withColumn("file_path",col("_metadata.file_path"))
                            .withColumn("ingestion_date",current_timestamp())
)

In [0]:
streaming_query = (customer_transformed_df.writeStream \
    .trigger(availableNow=True) \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/udemydatabricks/dataengg/streaming_volume/checkpoints/customer_stream/_checkpoint_stream") \
    .toTable("udemydatabricks.dataengg.customer_stream"))

In [0]:

# customer_stream = (
#     spark.readStream
#          .format("delta")
#          .table("udemydatabricks.dataengg.customer")
# )
# display(customer_stream, checkpointLocation="/Volumes/udemydatabricks/dataengg/streaming_volume/checkpoints/customer_stream")
